# 01 Data audit and cleaning


## 1. Parse the raw file
Read every column as text (keeps 13-digit sensor IDs exact), unpack the JSON into one row per (sensor, time, variable), drop sensor-flagged `errorCode` readings and repeated uploads.

In [1]:
from mma3001_project.load import parse_env_csv, sensor_summary, clean_sensor_frame, cleaning_report
long = parse_env_csv("../data/raw/5EnvSensor_MayToDec2024_180kRows.csv")  # long format
long.shape

(430862, 5)

## 2. Check sensor acitivity
`unique_ratio` = unique measurement times / uploaded rows. Frozen sensors re-send one cached reading, so their ratio is ≈ 0.

In [2]:
sensor_summary(long)

,n_uploads,n_readings,first,last,n_distinct_co2,unique_ratio,is_live
sensorid,,,,,,,
6012002000227,21363,1,2023-12-11 05:07:53+00:00,2023-12-11 05:07:53+00:00,1,0.000,False
6012002000241,58908,2,2023-12-11 05:52:45+00:00,2025-05-19 13:08:04+00:00,2,0.000,False
6012002000326,58909,29039,2023-12-11 05:00:57+00:00,2026-04-22 13:12:41+00:00,245,0.493,True
6012002000777,10926,1,2023-05-05 22:42:45+00:00,2023-05-05 22:42:45+00:00,1,0.000,False
6012002000869,29341,13800,2024-02-29 06:04:56+00:00,2024-06-13 17:45:44+00:00,222,0.470,True


**Result:** only sensors 326 and 869 are live (ratio ≈ 0.5); the other three repeat 1–2 values.

## 3. Clean sensor 869
Sensor 869 is used because it is live and sits in the identified room. Values outside physical ranges become NaN (`load.VALID_RANGES`), then readings are binned to a 10-min grid.

In [3]:
clean = clean_sensor_frame(long, "6012002000869", end="2024-06-14")   # last reading: 13 Jun 2024
clean.head()

,co2_ppm,temp_c,rh_pct
time_utc,,,
2024-02-29 06:00:00+00:00,465.0,24.3,56.0
2024-02-29 06:10:00+00:00,445.0,24.0,56.0
2024-02-29 06:20:00+00:00,NaN,NaN,NaN
2024-02-29 06:30:00+00:00,471.0,23.8,56.0
2024-02-29 06:40:00+00:00,457.0,23.8,56.0


## 4. Data-quality check

In [4]:
cleaning_report(clean)

,slots,missing,missing_pct,n_gaps,longest_gap,min,median,max
co2_ppm,15191.0,1391.0,9.2,1390.0,2.0,403.0,463.0,646.0
temp_c,15191.0,1391.0,9.2,1390.0,2.0,19.0,23.3,28.5
rh_pct,15191.0,1391.0,9.2,1390.0,2.0,26.0,45.0,75.0


**Result:** 9.2 % of slots are missing, every gap is 1–2 slots (≤ 20 min), and all values lie inside the valid ranges (CO₂ 403–646 ppm, 19.0–28.5 °C, RH 26–75 %). Short gaps justify interpolation in 02/03.

## 5. Save

In [5]:
import os
os.makedirs("../data/processed", exist_ok=True)
clean.to_csv("../data/processed/sensor869_clean.csv")